<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl_dogfight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q stable-baselines3 gymnasium
!pip install -q jsbsim==1.2.4
!pip install -q pyyaml
!pip install -q optuna

import jsbsim
print(jsbsim.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 19.8 MB/s eta 0:00:00
1.2.4


In [2]:
!pip install fastapi uvicorn websockets --quiet


In [3]:
#repodaki güncel dosyaları çeker PUSHLAMADAN ÇALIŞTIRMA
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.			   .gitignore		      README.md
..			   notebooks		      requirements.txt
configs			   ppo_final.acmi	      runs
dogfightSim_realtime.html  ppo_flight_final.acmi      sac_final.acmi
droneSim_realtime.html	   ppo_flight_telemetry.csv   src
f450-drone-framestl.stl    ppo_telemetry.csv	      telemetry_for_html.json
.git			   ppo_vs_sac_comparison.png


In [4]:
import torch
print("torch OK:", torch.__version__)
import stable_baselines3
print("sb3 OK:", stable_baselines3.__version__)
import jsbsim
print("jsbsim OK:", jsbsim.__version__)
import fastapi, uvicorn
print("fastapi/uvicorn OK")

torch OK: 2.11.0+cpu


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


sb3 OK: 2.9.0
jsbsim OK: 1.2.4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


fastapi/uvicorn OK


In [5]:
import os
os.chdir('/content')
print(os.getcwd())

/content


In [6]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [7]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/dogfight_stage_a.yaml
/content/repo/configs/dogfight_stage_b.yaml
/content/repo/configs/ppo_flight_stage1.yaml
/content/repo/configs/ppo_flight.yaml
/content/repo/configs/ppo_hover_customnet.yaml
/content/repo/configs/ppo_hover.yaml
/content/repo/dogfightSim_realtime.html
/content/repo/droneSim_realtime.html
/content/repo/f450-drone-framestl.stl
/content/repo/.gitignore
/content/repo/notebooks/quadcopter_rl_dogfight.ipynb
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/ppo_final.acmi
/content/repo/ppo_flight_final.acmi
/content/repo/ppo_flight_telemetry.csv
/content/repo/ppo_telemetry.csv
/content/repo/ppo_vs_sac_comparison.png
/content/repo/README.md
/content/repo/requirements.txt
/content/repo/runs/dogfight_pool/manifest.json
/content/repo/runs/dogfight_pool/v1/model.zip
/content/repo/runs/dogfight_pool/v1/vecnormalize.pkl
/content/repo/runs/dogfight_stage_a/model_final.zip
/content/repo/runs/dogfight_stage_a/vecnormalize.pkl
/content/repo/runs/dogfigh

In [91]:
#pull gerekmiyorsa git push
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add .

git commit -m "kaybedilen kodlar"
git pull origin main --no-edit
git push origin main

[reload] TRAINING modeli guncellendi


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[main bc58664] kaybedilen kodlar
 18 files changed, 3647 insertions(+), 192 deletions(-)
 create mode 100644 main.py
 create mode 100644 src/drone_rl/dogfight/__init__.py
 create mode 100644 train_a_log.txt
 create mode 100644 train_b_log.txt
Already up to date.


From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   76fc738..bc58664  main -> main


In [ ]:
#pull gerekiyorsa git push
%%bash
cd /content/repo

# Önce Colab'daki mevcut değişiklikleri geçici olarak sakla
git stash push -u -m "colab-local-changes"

# GitHub'daki güncel hali al
git pull origin main --no-rebase

# Colab'daki değişiklikleri geri getir
git stash pop

# Değişiklikleri commit et
git add .
git commit -m "ppo aşamalı training ve simülasyon kontrol fix"

# GitHub'a gönder
git push origin main

No local changes to save
Merge made by the 'ort' strategy.
 notebooks/quadcopter_rl.ipynb | 8525 +++++++++++++++++++++--------------------
 1 file changed, 4386 insertions(+), 4139 deletions(-)
On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
No stash entries found.
To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   89cfcc0..1cc330c  main -> main


In [17]:
%%writefile /content/repo/src/drone_rl/dogfight/config.py
from dataclasses import dataclass, field
from typing import Optional, List
import yaml


@dataclass
class DogfightEnvConfig:
    episode_seconds: float = 45.0
    physics_hz: int = 240
    control_hz: int = 20
    hover_throttle: float = 0.420
    throttle_range: float = 0.25

    roll_authority: float = 0.6
    pitch_authority: float = 0.6
    yaw_authority: float = 0.45
    control_surface_tau_s: float = 0.08

    cone_half_angle_deg: float = 30.0
    cone_range_ft: float = 70.0  # v4: 120'den kisaltildi (gorsel + oyun mantigi)

    standoff_target_ft: float = 40.0
    standoff_weight_start: float = 0.02
    standoff_weight_end: float = 0.08
    standoff_ramp_steps: int = 400_000
    standoff_penalty_cap: float = 2.0

    reward_align_weight: float = 0.20
    reward_exposure_weight: float = 0.15
    reward_cone_hold: float = 0.08
    reward_tilt_weight: float = 0.03
    reward_spin_weight: float = 0.06
    reward_yawrate_weight: float = 0.04
    reward_jerk_weight: float = 0.05
    opponent_fault_bonus: float = 5.0

    crash_penalty: float = 30.0
    crash_min_alt_ft: float = 5.0
    crash_max_alt_ft: float = 250.0
    crash_max_tilt_rad: float = 0.7
    crash_max_yawrate_rps: float = 20.0
    max_horizontal_range_ft: float = 220.0
    min_separation_ft: float = 10.0

    base_altitude_ft: float = 150.0
    altitude_jitter_ft: float = 15.0
    spawn_range_min_ft: float = 60.0
    spawn_range_max_ft: float = 150.0

    opponent_latest_prob: float = 0.7


@dataclass
class PPOConfig:
    policy: str = "MlpPolicy"
    n_steps: int = 2048
    batch_size: int = 256
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    learning_rate: float = 3e-4
    ent_coef: float = 0.0
    net_arch_pi: Optional[List[int]] = None
    net_arch_vf: Optional[List[int]] = None
    activation_fn: Optional[str] = None


@dataclass
class TrainConfig:
    timesteps: int = 500_000
    n_envs: int = 4


@dataclass
class PromotionConfig:
    eval_freq: int = 30_000
    n_eval_episodes: int = 30
    win_rate_threshold: float = 0.55
    mean_reward_improve_pct: float = 8.0
    consecutive_passes_required: int = 3


@dataclass
class DogfightConfig:
    env: DogfightEnvConfig = field(default_factory=DogfightEnvConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    promotion: PromotionConfig = field(default_factory=PromotionConfig)


def load_dogfight_config(path: Optional[str]) -> DogfightConfig:
    if path is None:
        return DogfightConfig()
    with open(path, "r") as f:
        raw = yaml.safe_load(f) or {}
    return DogfightConfig(
        env=DogfightEnvConfig(**raw.get("env", {})),
        ppo=PPOConfig(**raw.get("ppo", {})),
        train=TrainConfig(**raw.get("train", {})),
        promotion=PromotionConfig(**raw.get("promotion", {})),
    )

Overwriting /content/repo/src/drone_rl/dogfight/config.py


In [18]:
%%writefile /content/repo/src/drone_rl/dogfight/__init__.py

Writing /content/repo/src/drone_rl/dogfight/__init__.py


In [19]:
%%writefile /content/repo/src/drone_rl/dogfight/checkpoint_pool.py
import json
import shutil
from pathlib import Path
from typing import Optional, Tuple

import numpy as np


class CheckpointPool:
    def __init__(self, pool_dir: str):
        self.pool_dir = Path(pool_dir)
        self.pool_dir.mkdir(parents=True, exist_ok=True)
        self.manifest_path = self.pool_dir / "manifest.json"
        self._load()

    def _load(self):
        if self.manifest_path.exists():
            self.entries = json.loads(self.manifest_path.read_text())
        else:
            self.entries = []

    def _save(self):
        self.manifest_path.write_text(json.dumps(self.entries, indent=2))

    def __len__(self):
        return len(self.entries)

    def add(self, model_src: str, vecnorm_src: str, mean_reward: float,
            win_rate: float, note: str = "") -> int:
        version = len(self.entries) + 1
        dst_dir = self.pool_dir / f"v{version}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        model_dst = dst_dir / "model.zip"
        vecnorm_dst = dst_dir / "vecnormalize.pkl"
        shutil.copy(model_src, model_dst)
        shutil.copy(vecnorm_src, vecnorm_dst)
        entry = {
            "version": version,
            "model": str(model_dst),
            "vecnorm": str(vecnorm_dst),
            "mean_reward": float(mean_reward),
            "win_rate": float(win_rate),
            "note": note,
        }
        self.entries.append(entry)
        self._save()
        return version

    def latest(self) -> Optional[Tuple[str, str]]:
        if not self.entries:
            return None
        e = self.entries[-1]
        return e["model"], e["vecnorm"]

    def latest_mean_reward(self) -> Optional[float]:
        if not self.entries:
            return None
        return self.entries[-1]["mean_reward"]

    def sample(self, latest_prob: float = 0.7, rng: Optional[np.random.Generator] = None
               ) -> Optional[Tuple[str, str]]:
        if not self.entries:
            return None
        rng = rng or np.random.default_rng()
        if len(self.entries) == 1 or rng.random() < latest_prob:
            e = self.entries[-1]
        else:
            idx = int(rng.integers(0, len(self.entries) - 1))
            e = self.entries[idx]
        return e["model"], e["vecnorm"]

    def summary(self) -> str:
        lines = [f"Pool: {self.pool_dir} ({len(self.entries)} versiyon)"]
        for e in self.entries:
            lines.append(f"  v{e['version']}: mean_reward={e['mean_reward']:.2f} "
                          f"win_rate={e['win_rate']:.2f} note={e['note']}")
        return "\n".join(lines)

Overwriting /content/repo/src/drone_rl/dogfight/checkpoint_pool.py


In [20]:
%%writefile /content/repo/src/drone_rl/dogfight/dogfight_env.py
"""Iki F450 arasinda 'dogfight' gorevi - birbirini kovalayip radar
konisine alma. TAM 3D fizik.

v5: Stage B'de rakip HER reset()'te havuzdan yeniden orneklenir.
v4: KRITIK konum duzeltmesi - mutlak enlem/boylam (ic/lat-gc-deg,
    ic/long-gc-deg) kullaniliyor. Eskiden "distance-from-start-*"
    property'leri kullaniliyordu, bunlar HER FDM'IN KENDI baslangicina
    gore olcum yapiyordu (iki FDM arasi PAYLASILAN referans DEGIL) -
    iki drone pratikte HEP ayni noktada spawn oluyordu. Ampirik JSBSim
    testiyle dogrulanan duzeltme.
v3: info dict'e her iki drone icin kinematik + odul kirilimi eklendi.

--- KIM RL ILE CALISIYOR? ---
- fdm_self: HER ZAMAN dis taraftan (PPO) gelen action ile suruluyor.
- fdm_opp: self.opponent_controller uzerinden - Stage A'da scripted
  (RL degil), Stage B'de dondurulmus/inference-only bir PPO modeli.
- reward SADECE fdm_self icin hesaplanir, opp hicbir zaman bu adimda
  ogrenmez (frozen opponent self-play deseni).
"""

import math
from typing import Optional

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim

LAT0_DEG = 0.0
LON0_DEG = 0.0
FT_PER_DEG_LAT = 364567.2


class BaseOpponentController:
    def reset(self):
        pass

    def compute_action(self, env: "DogfightEnv") -> np.ndarray:
        raise NotImplementedError


class ScriptedCircleOpponent(BaseOpponentController):
    def __init__(self, bank_deg=15.0, kp_roll=3.5, kd_roll=0.3, kp_alt=0.12, kd_alt=0.35):
        self.bank_deg = bank_deg
        self.kp_roll = kp_roll
        self.kd_roll = kd_roll
        self.kp_alt = kp_alt
        self.kd_alt = kd_alt
        self._target_alt_ft = None

    def reset(self):
        self._target_alt_ft = None

    def compute_action(self, env: "DogfightEnv") -> np.ndarray:
        f = env.fdm_opp
        if self._target_alt_ft is None:
            self._target_alt_ft = f["position/h-agl-ft"]

        roll_now = f["attitude/phi-rad"]
        p_now = f["velocities/p-rad_sec"]
        target_roll = math.radians(self.bank_deg)
        roll_cmd = float(np.clip(
            self.kp_roll * (target_roll - roll_now) - self.kd_roll * p_now, -1.0, 1.0
        ))
        pitch_cmd = 0.0

        alt_err = self._target_alt_ft - f["position/h-agl-ft"]
        hdot = f["velocities/h-dot-fps"]
        throttle_cmd = float(np.clip(self.kp_alt * alt_err - self.kd_alt * hdot, -1.0, 1.0))
        yaw_cmd = 0.0

        return np.array([roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd], dtype=np.float32)


class NormalizerStats:
    def __init__(self, vecnorm):
        self.mean = vecnorm.obs_rms.mean.astype(np.float32)
        self.var = vecnorm.obs_rms.var.astype(np.float32)
        self.epsilon = vecnorm.epsilon
        self.clip_obs = vecnorm.clip_obs

    def normalize(self, obs: np.ndarray) -> np.ndarray:
        normed = (obs - self.mean) / np.sqrt(self.var + self.epsilon)
        return np.clip(normed, -self.clip_obs, self.clip_obs).astype(np.float32)


class PPOOpponentController(BaseOpponentController):
    def __init__(self, model, stats: NormalizerStats):
        self.model = model
        self.stats = stats

    def compute_action(self, env: "DogfightEnv") -> np.ndarray:
        obs = env._get_obs_for(env.fdm_opp, env.fdm_self, env.prev_action_opp)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=True)
        return action[0]


class DogfightEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, cfg, opponent_controller: Optional[BaseOpponentController] = None,
                 opponent_pool=None, opponent_latest_prob: float = 0.7):
        super().__init__()
        self.cfg = cfg

        self.physics_hz = int(cfg.physics_hz)
        self.control_hz = int(cfg.control_hz)
        self.physics_dt = 1.0 / self.physics_hz
        self.substeps = self.physics_hz // self.control_hz
        self.max_steps = int(cfg.episode_seconds * self.control_hz)

        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(17,), dtype=np.float32)

        self.fdm_self = jsbsim.FGFDMExec(None)
        self.fdm_self.set_debug_level(0)
        if not self.fdm_self.load_model("F450"):
            raise RuntimeError("F450 (self) yuklenemedi")
        self.fdm_self.set_dt(self.physics_dt)

        self.fdm_opp = jsbsim.FGFDMExec(None)
        self.fdm_opp.set_debug_level(0)
        if not self.fdm_opp.load_model("F450"):
            raise RuntimeError("F450 (opp) yuklenemedi")
        self.fdm_opp.set_dt(self.physics_dt)

        self.opponent_controller = opponent_controller or ScriptedCircleOpponent()
        self.opponent_pool = opponent_pool
        self.opponent_latest_prob = opponent_latest_prob

        self._surface_self = np.zeros(3, dtype=np.float64)
        self._surface_opp = np.zeros(3, dtype=np.float64)
        self.prev_action_self = np.zeros(4, dtype=np.float32)
        self.prev_action_opp = np.zeros(4, dtype=np.float32)

        self.step_count = 0
        self.my_score = 0
        self.opp_score = 0
        self._prev_range_ft = None

        self.standoff_weight = cfg.standoff_weight_start

    @property
    def control_dt(self):
        return self.substeps * self.physics_dt

    def set_standoff_weight(self, w: float):
        self.standoff_weight = float(w)

    def set_opponent_controller(self, controller: BaseOpponentController):
        self.opponent_controller = controller

    def set_opponent_controller_from_pool(self, model_vecnorm_tuple):
        from drone_rl.dogfight.env_factory import load_opponent_controller
        model_path, vecnorm_path = model_vecnorm_tuple
        self.opponent_controller = load_opponent_controller(model_path, vecnorm_path)

    def _init_fdm(self, fdm, alt_ft, heading_deg, north_ft=0.0, east_ft=0.0):
        ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
        fdm["ic/lat-gc-deg"] = LAT0_DEG + north_ft / FT_PER_DEG_LAT
        fdm["ic/long-gc-deg"] = LON0_DEG + east_ft / ft_per_deg_lon
        fdm["ic/h-agl-ft"] = alt_ft
        fdm["ic/u-fps"] = 0.0
        fdm["ic/v-fps"] = 0.0
        fdm["ic/w-fps"] = 0.0
        fdm["ic/phi-rad"] = 0.0
        fdm["ic/theta-rad"] = 0.0
        fdm["ic/psi-true-rad"] = math.radians(heading_deg)
        fdm.run_ic()
        for i in range(4):
            fdm[f"propulsion/engine[{i}]/set-running"] = 1
        fdm["fcs/ScasEngage"] = 1
        fdm["fcs/aileron-cmd-norm"] = 0.0
        fdm["fcs/elevator-cmd-norm"] = 0.0
        fdm["fcs/rudder-cmd-norm"] = 0.0
        fdm["fcs/throttle-cmd-norm"] = self.cfg.hover_throttle

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        cfg = self.cfg

        if self.opponent_pool is not None:
            sampled = self.opponent_pool.sample(self.opponent_latest_prob, rng=self.np_random)
            if sampled is not None:
                from drone_rl.dogfight.env_factory import load_opponent_controller
                self.opponent_controller = load_opponent_controller(*sampled)

        rng = self.np_random
        rng_range = rng.uniform(cfg.spawn_range_min_ft, cfg.spawn_range_max_ft)
        bearing_deg = rng.uniform(0.0, 360.0)
        alt_self = cfg.base_altitude_ft + rng.uniform(-cfg.altitude_jitter_ft, cfg.altitude_jitter_ft)
        alt_opp = cfg.base_altitude_ft + rng.uniform(-cfg.altitude_jitter_ft, cfg.altitude_jitter_ft)
        heading_self = rng.uniform(0.0, 360.0)
        heading_opp = rng.uniform(0.0, 360.0)

        dn_target = rng_range * math.cos(math.radians(bearing_deg))
        de_target = rng_range * math.sin(math.radians(bearing_deg))

        self._init_fdm(self.fdm_self, alt_self, heading_self, north_ft=0.0, east_ft=0.0)
        self._init_fdm(self.fdm_opp, alt_opp, heading_opp, north_ft=dn_target, east_ft=de_target)

        self._surface_self[:] = 0.0
        self._surface_opp[:] = 0.0
        self.prev_action_self = np.zeros(4, dtype=np.float32)
        self.prev_action_opp = np.zeros(4, dtype=np.float32)
        self.step_count = 0
        self.my_score = 0
        self.opp_score = 0
        self.opponent_controller.reset()

        rng_ft, _, _, _, _ = self._relative_geom(self.fdm_self, self.fdm_opp)
        self._prev_range_ft = rng_ft

        return self._get_obs_for(self.fdm_self, self.fdm_opp, self.prev_action_self), {}

    def _nose_vector(self, fdm):
        psi = fdm["attitude/psi-rad"]
        theta = fdm["attitude/theta-rad"]
        n = math.cos(theta) * math.cos(psi)
        e = math.cos(theta) * math.sin(psi)
        u = math.sin(theta)
        return n, e, u

    def _position(self, fdm):
        lat = fdm["position/lat-gc-deg"]
        lon = fdm["position/long-gc-deg"]
        ft_per_deg_lon = FT_PER_DEG_LAT * math.cos(math.radians(LAT0_DEG))
        north_ft = (lat - LAT0_DEG) * FT_PER_DEG_LAT
        east_ft = (lon - LON0_DEG) * ft_per_deg_lon
        alt_ft = fdm["position/h-agl-ft"]
        return north_ft, east_ft, alt_ft

    def _relative_geom(self, fdm_a, fdm_b):
        na, ea, ua = self._position(fdm_a)
        nb, eb, ub = self._position(fdm_b)
        dn, de, dz = nb - na, eb - ea, ub - ua
        rng_ft = max(math.sqrt(dn * dn + de * de + dz * dz), 1e-3)
        los = (dn / rng_ft, de / rng_ft, dz / rng_ft)
        nose = self._nose_vector(fdm_a)
        align_cos = nose[0] * los[0] + nose[1] * los[1] + nose[2] * los[2]
        return rng_ft, align_cos, dn, de, dz

    def _get_obs_for(self, fdm_owner, fdm_other, prev_action_owner):
        roll = fdm_owner["attitude/phi-rad"]
        pitch = fdm_owner["attitude/theta-rad"]
        p = fdm_owner["velocities/p-rad_sec"] / 5.0
        q = fdm_owner["velocities/q-rad_sec"] / 5.0
        r = fdm_owner["velocities/r-rad_sec"] / 5.0
        hdot = fdm_owner["velocities/h-dot-fps"] / 10.0

        rng_ft, align_owner, dn, de, dz = self._relative_geom(fdm_owner, fdm_other)
        _, align_other, _, _, _ = self._relative_geom(fdm_other, fdm_owner)

        if fdm_owner is self.fdm_self:
            closing_fps = (self._prev_range_ft - rng_ft) / self.control_dt if self._prev_range_ft else 0.0
        else:
            closing_fps = 0.0

        range_n = rng_ft / 100.0
        closing_n = closing_fps / 20.0
        dz_n = dz / 50.0

        cone_half_cos = math.cos(math.radians(self.cfg.cone_half_angle_deg))
        other_in_owner_cone = 1.0 if (align_owner >= cone_half_cos and rng_ft <= self.cfg.cone_range_ft) else 0.0
        owner_in_other_cone = 1.0 if (align_other >= cone_half_cos and rng_ft <= self.cfg.cone_range_ft) else 0.0

        return np.array([
            roll, pitch, p, q, r, hdot,
            range_n, closing_n,
            align_owner, align_other,
            dz_n,
            other_in_owner_cone, owner_in_other_cone,
            *prev_action_owner,
        ], dtype=np.float32)

    def _apply_action(self, fdm, surface_state, action):
        roll_cmd, pitch_cmd, yaw_cmd, throttle_cmd = action
        aileron_t = float(np.clip(roll_cmd * self.cfg.roll_authority, -1.0, 1.0))
        elevator_t = float(np.clip(-pitch_cmd * self.cfg.pitch_authority, -1.0, 1.0))
        rudder_t = float(np.clip(yaw_cmd * self.cfg.yaw_authority, -1.0, 1.0))
        throttle = float(np.clip(
            self.cfg.hover_throttle + throttle_cmd * self.cfg.throttle_range, 0.0, 1.0
        ))
        targets = np.array([aileron_t, elevator_t, rudder_t])
        alpha = self.physics_dt / (self.cfg.control_surface_tau_s + self.physics_dt)
        surface_state += alpha * (targets - surface_state)
        fdm["fcs/aileron-cmd-norm"] = float(surface_state[0])
        fdm["fcs/elevator-cmd-norm"] = float(surface_state[1])
        fdm["fcs/rudder-cmd-norm"] = float(surface_state[2])
        fdm["fcs/throttle-cmd-norm"] = throttle

    def _is_out_of_bounds(self, fdm):
        alt = fdm["position/h-agl-ft"]
        n_ft, e_ft, _ = self._position(fdm)
        horiz = math.hypot(n_ft, e_ft)
        yaw_rate = abs(fdm["velocities/r-rad_sec"])
        return (
            alt < self.cfg.crash_min_alt_ft
            or alt > self.cfg.crash_max_alt_ft
            or abs(fdm["attitude/phi-rad"]) > self.cfg.crash_max_tilt_rad
            or abs(fdm["attitude/theta-rad"]) > self.cfg.crash_max_tilt_rad
            or yaw_rate > self.cfg.crash_max_yawrate_rps
            or horiz > self.cfg.max_horizontal_range_ft
        )

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(4)
        opp_action = self.opponent_controller.compute_action(self)
        opp_action = np.asarray(opp_action, dtype=np.float32).reshape(4)

        for _ in range(self.substeps):
            self._apply_action(self.fdm_self, self._surface_self, action)
            self._apply_action(self.fdm_opp, self._surface_opp, opp_action)
            self.fdm_self.run()
            self.fdm_opp.run()

        self.step_count += 1

        rng_ft, align_mine, dn, de, dz = self._relative_geom(self.fdm_self, self.fdm_opp)
        _, align_opp_to_me, _, _, _ = self._relative_geom(self.fdm_opp, self.fdm_self)
        closing_fps = (self._prev_range_ft - rng_ft) / self.control_dt
        self._prev_range_ft = rng_ft

        cone_half_cos = math.cos(math.radians(self.cfg.cone_half_angle_deg))
        opp_in_my_cone = align_mine >= cone_half_cos and rng_ft <= self.cfg.cone_range_ft
        me_in_opp_cone = align_opp_to_me >= cone_half_cos and rng_ft <= self.cfg.cone_range_ft

        align_reward = self.cfg.reward_align_weight * align_mine
        exposure_penalty = self.cfg.reward_exposure_weight * align_opp_to_me
        standoff_raw = ((rng_ft - self.cfg.standoff_target_ft) / self.cfg.standoff_target_ft) ** 2
        standoff_penalty = self.standoff_weight * min(standoff_raw, self.cfg.standoff_penalty_cap)

        tilt = abs(self.fdm_self["attitude/phi-rad"]) + abs(self.fdm_self["attitude/theta-rad"])
        spin = abs(self.fdm_self["velocities/p-rad_sec"]) + abs(self.fdm_self["velocities/q-rad_sec"])
        yaw_rate_pen = abs(self.fdm_self["velocities/r-rad_sec"])
        jerk = float(np.sum(np.abs(action - self.prev_action_self)))
        control_penalty = (
            self.cfg.reward_tilt_weight * tilt
            + self.cfg.reward_spin_weight * spin
            + self.cfg.reward_yawrate_weight * yaw_rate_pen
            + self.cfg.reward_jerk_weight * jerk
        )

        opp_tilt = abs(self.fdm_opp["attitude/phi-rad"]) + abs(self.fdm_opp["attitude/theta-rad"])
        opp_spin = abs(self.fdm_opp["velocities/p-rad_sec"]) + abs(self.fdm_opp["velocities/q-rad_sec"])
        opp_yaw_rate_pen = abs(self.fdm_opp["velocities/r-rad_sec"])
        opp_jerk = float(np.sum(np.abs(opp_action - self.prev_action_opp)))
        opp_control_penalty = (
            self.cfg.reward_tilt_weight * opp_tilt
            + self.cfg.reward_spin_weight * opp_spin
            + self.cfg.reward_yawrate_weight * opp_yaw_rate_pen
            + self.cfg.reward_jerk_weight * opp_jerk
        )
        opp_align_reward = self.cfg.reward_align_weight * align_opp_to_me
        opp_exposure_penalty = self.cfg.reward_exposure_weight * align_mine

        cone_net = 0.0
        if opp_in_my_cone:
            cone_net += self.cfg.reward_cone_hold
            self.my_score += 1
        if me_in_opp_cone:
            cone_net -= self.cfg.reward_cone_hold
            self.opp_score += 1

        reward = align_reward - exposure_penalty - standoff_penalty - control_penalty + cone_net

        self_oob = self._is_out_of_bounds(self.fdm_self)
        opp_oob = self._is_out_of_bounds(self.fdm_opp)
        collided = rng_ft < self.cfg.min_separation_ft

        crashed = False
        terminated = False
        reset_reason = None
        if collided:
            reward -= self.cfg.crash_penalty
            crashed = True
            terminated = True
            reset_reason = "collision"
        elif self_oob:
            reward -= self.cfg.crash_penalty
            crashed = True
            terminated = True
            reset_reason = "self_crash"
        elif opp_oob:
            reward += self.cfg.opponent_fault_bonus
            terminated = True
            reset_reason = "opponent_crash"

        self.prev_action_self = action.copy()
        self.prev_action_opp = opp_action.copy()

        truncated = bool(self.step_count >= self.max_steps)
        if truncated and reset_reason is None:
            reset_reason = "timeout"

        obs = self._get_obs_for(self.fdm_self, self.fdm_opp, self.prev_action_self)

        info = {
            "range_ft": rng_ft,
            "closing_fps": closing_fps,
            "align_mine": align_mine,
            "align_opp": align_opp_to_me,
            "opp_in_my_cone": bool(opp_in_my_cone),
            "me_in_opp_cone": bool(me_in_opp_cone),
            "my_score": self.my_score,
            "opp_score": self.opp_score,
            "crashed": crashed,
            "reset_reason": reset_reason,
            "align_reward": align_reward,
            "exposure_penalty": exposure_penalty,
            "standoff_penalty": standoff_penalty,
            "control_penalty": control_penalty,
            "cone_net": cone_net,
            "opp_align_reward": opp_align_reward,
            "opp_exposure_penalty": opp_exposure_penalty,
            "opp_standoff_penalty": standoff_penalty,
            "opp_control_penalty": opp_control_penalty,
            "self_hdot_fps": float(self.fdm_self["velocities/h-dot-fps"]),
            "opp_hdot_fps": float(self.fdm_opp["velocities/h-dot-fps"]),
            "self_pos": self._position(self.fdm_self),
            "opp_pos": self._position(self.fdm_opp),
            "self_attitude": (
                self.fdm_self["attitude/phi-rad"],
                self.fdm_self["attitude/theta-rad"],
                self.fdm_self["attitude/psi-rad"],
            ),
            "opp_attitude": (
                self.fdm_opp["attitude/phi-rad"],
                self.fdm_opp["attitude/theta-rad"],
                self.fdm_opp["attitude/psi-rad"],
            ),
        }

        return obs, float(reward), terminated, truncated, info

Overwriting /content/repo/src/drone_rl/dogfight/dogfight_env.py


In [58]:
%%writefile /content/repo/src/drone_rl/dogfight/env_factory.py
"""Dogfight env/VecEnv kurulum yardimcilari.

DUZELTME: load_opponent_controller() eskiden HER cagrildiginda gercek
bir DogfightEnv (2 JSBSim FDM'i) kuruyordu - sadece VecNormalize'i
yuklemek icin. "Her episode'da havuzdan yeniden ornekleme" ozelligiyle
birlikte bu, saniyede onlarca kez GEREKSIZ JSBSim motoru kurulup
atilmasina yol aciyordu - hem logu spam'liyor hem egitimi ciddi
yavaslatiyordu. Simdi:
  1. VecNormalize icin GERCEK JSBSim GEREKTIRMEYEN, sadece observation/
     action_space'i eslesen SAHTE bir env kullaniliyor.
  2. Ayni (model_path, vecnorm_path) tekrar istenirse ONBELLEKTEN
     donduruluyor (ki bu COK sik oluyor, %70 ihtimalle hep en guncel
     checkpoint isteniyor).
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from drone_rl.dogfight.dogfight_env import (
    DogfightEnv, ScriptedCircleOpponent, PPOOpponentController, NormalizerStats,
)
from drone_rl.dogfight.checkpoint_pool import CheckpointPool


class _DummyObsEnv(gym.Env):
    """VecNormalize.load() icin JSBSim GEREKTIRMEYEN, sadece observation/
    action_space uyumlu sahte bir env. Gercek fizik hic calismiyor -
    tek amaci VecNormalize'in bir Venv'e 'baglanabilmesi' icin bir
    yer tutucu olmak."""

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(17,), dtype=np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        return np.zeros(17, dtype=np.float32), {}

    def step(self, action):
        return np.zeros(17, dtype=np.float32), 0.0, True, False, {}


# YENI: ayni (model_path, vecnorm_path) tekrar istenirse tekrar
# yuklemek yerine onbellekten donduruluyor.
_controller_cache: dict = {}


def load_opponent_controller(model_path: str, vecnorm_path: str):
    cache_key = (model_path, vecnorm_path)
    if cache_key in _controller_cache:
        return _controller_cache[cache_key]

    from stable_baselines3 import PPO
    dummy_venv = DummyVecEnv([lambda: Monitor(_DummyObsEnv())])
    vecnorm = VecNormalize.load(vecnorm_path, dummy_venv)
    stats = NormalizerStats(vecnorm)
    model = PPO.load(model_path, device="cpu")

    controller = PPOOpponentController(model, stats)
    _controller_cache[cache_key] = controller
    return controller


def make_dogfight_env(env_cfg, stage: str = "a", pool_dir: str = None,
                       fixed_opponent=None) -> DogfightEnv:
    if fixed_opponent is not None:
        controller = load_opponent_controller(*fixed_opponent)
        return DogfightEnv(env_cfg, opponent_controller=controller)

    if stage == "b" and pool_dir:
        pool = CheckpointPool(pool_dir)
        return DogfightEnv(
            env_cfg,
            opponent_controller=ScriptedCircleOpponent(),
            opponent_pool=pool,
            opponent_latest_prob=env_cfg.opponent_latest_prob,
        )

    return DogfightEnv(env_cfg, opponent_controller=ScriptedCircleOpponent())


def make_dogfight_training_vec_env(env_cfg, n_envs: int, stage: str, pool_dir: str,
                                    training: bool, norm_reward: bool, clip_obs: float = 10.0):
    def _make():
        return Monitor(make_dogfight_env(env_cfg, stage=stage, pool_dir=pool_dir))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=norm_reward,
                         clip_obs=clip_obs, training=training)
    return venv

Overwriting /content/repo/src/drone_rl/dogfight/env_factory.py


In [22]:
%%writefile /content/repo/src/drone_rl/dogfight/train.py
import argparse
from pathlib import Path

import numpy as np
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.env_factory import make_dogfight_training_vec_env, make_dogfight_env
from drone_rl.dogfight.checkpoint_pool import CheckpointPool
from drone_rl.dogfight.dogfight_env import NormalizerStats


ACTIVATION_MAP = {"tanh": nn.Tanh, "relu": nn.ReLU}


def build_policy_kwargs(cfg_ppo):
    kwargs = {}
    pi_arch = cfg_ppo.net_arch_pi or [128, 128]
    vf_arch = cfg_ppo.net_arch_vf or [128, 128]
    kwargs["net_arch"] = dict(pi=pi_arch, vf=vf_arch)
    if cfg_ppo.activation_fn:
        kwargs["activation_fn"] = ACTIVATION_MAP[cfg_ppo.activation_fn.lower()]
    return kwargs


class StandoffCurriculumCallback(BaseCallback):
    def __init__(self, w_start, w_end, ramp_steps):
        super().__init__()
        self.w_start = w_start
        self.w_end = w_end
        self.ramp_steps = max(ramp_steps, 1)

    def _on_step(self) -> bool:
        frac = min(1.0, self.num_timesteps / self.ramp_steps)
        w = self.w_start + (self.w_end - self.w_start) * frac
        self.training_env.env_method("set_standoff_weight", w)
        return True


class PeriodicSnapshotCallback(BaseCallback):
    def __init__(self, out_dir: Path, save_freq: int):
        super().__init__()
        self.snapshot_dir = Path(out_dir) / "live_snapshot"
        self.snapshot_dir.mkdir(parents=True, exist_ok=True)
        self.save_freq = max(save_freq, 1)

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:
            vecnorm = self.model.get_vec_normalize_env()
            self.model.save(str(self.snapshot_dir / "model"))
            if vecnorm is not None:
                vecnorm.save(str(self.snapshot_dir / "vecnormalize.pkl"))
        return True


def _evaluate_vs_fixed(model, vecnorm_stats: NormalizerStats, env_cfg,
                        fixed_opponent, n_episodes: int):
    env = make_dogfight_env(env_cfg, fixed_opponent=fixed_opponent)
    rewards, wins = [], 0
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            norm_obs = vecnorm_stats.normalize(obs).reshape(1, -1)
            action, _ = model.predict(norm_obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action[0])
            ep_reward += reward
            done = terminated or truncated
        rewards.append(ep_reward)
        if info["my_score"] > info["opp_score"]:
            wins += 1
    return float(np.mean(rewards)), wins / n_episodes


class PromotionCallback(BaseCallback):
    def __init__(self, pool: CheckpointPool, env_cfg, promo_cfg, out_dir: Path):
        super().__init__()
        self.pool = pool
        self.env_cfg = env_cfg
        self.promo_cfg = promo_cfg
        self.out_dir = out_dir
        self._consecutive_pass = 0

    def _on_step(self) -> bool:
        if self.n_calls % self.promo_cfg.eval_freq != 0:
            return True

        latest = self.pool.latest()
        if latest is None:
            print("[promotion] havuz bos, atlaniyor")
            return True

        vecnorm_train = self.model.get_vec_normalize_env()
        stats = NormalizerStats(vecnorm_train)

        mean_reward, win_rate = _evaluate_vs_fixed(
            self.model, stats, self.env_cfg, latest, self.promo_cfg.n_eval_episodes
        )
        baseline = self.pool.latest_mean_reward() or 1e-6
        improve_pct = (mean_reward - baseline) / abs(baseline) * 100.0

        passed = (win_rate >= self.promo_cfg.win_rate_threshold
                  and improve_pct >= self.promo_cfg.mean_reward_improve_pct)

        print(f"[promotion] step={self.num_timesteps} win_rate={win_rate:.2f} "
              f"mean_reward={mean_reward:.2f} (baseline={baseline:.2f}, "
              f"improve={improve_pct:+.1f}%) pass={passed} "
              f"({self._consecutive_pass + (1 if passed else 0)}/"
              f"{self.promo_cfg.consecutive_passes_required})")

        if passed:
            self._consecutive_pass += 1
        else:
            self._consecutive_pass = 0

        if self._consecutive_pass >= self.promo_cfg.consecutive_passes_required:
            tmp_model = self.out_dir / f"_promo_candidate_{self.num_timesteps}.zip"
            tmp_vecnorm = self.out_dir / f"_promo_candidate_{self.num_timesteps}_vecnorm.pkl"
            self.model.save(str(tmp_model))
            vecnorm_train.save(str(tmp_vecnorm))

            version = self.pool.add(str(tmp_model), str(tmp_vecnorm), mean_reward, win_rate,
                                     note=f"step={self.num_timesteps}")
            print(f"[promotion] YENI VERSIYON: v{version} havuza eklendi!")

            tmp_model.unlink(missing_ok=True)
            tmp_vecnorm.unlink(missing_ok=True)

            new_latest = self.pool.latest()
            self.training_env.env_method("set_opponent_controller_from_pool", new_latest)
            self._consecutive_pass = 0

        return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", type=str, choices=["a", "b"])
    ap.add_argument("--config", type=str)
    ap.add_argument("--out", type=str, default="/content/runs/dogfight_run")
    ap.add_argument("--pool", type=str, default=None)
    ap.add_argument("--timesteps", type=int, default=None)
    ap.add_argument("--n-envs", type=int, default=None)
    ap.add_argument("--snapshot-freq", type=int, default=10000)

    ap.add_argument("--seed-pool", action="store_true")
    ap.add_argument("--from", dest="from_run", type=str)

    args = ap.parse_args()

    if args.seed_pool:
        if not args.from_run or not args.pool:
            raise ValueError("--seed-pool icin --from ve --pool gerekli")
        pool = CheckpointPool(args.pool)
        run_dir = Path(args.from_run)
        model_path = run_dir / "model_final.zip"
        vecnorm_path = run_dir / "vecnormalize.pkl"

        cfg = load_dogfight_config(None)
        from stable_baselines3.common.vec_env import DummyVecEnv
        from stable_baselines3.common.monitor import Monitor
        from drone_rl.dogfight.dogfight_env import DogfightEnv
        dummy = DummyVecEnv([lambda: Monitor(DogfightEnv(cfg.env))])
        vecnorm = VecNormalize.load(str(vecnorm_path), dummy)
        stats = NormalizerStats(vecnorm)
        model = PPO.load(str(model_path), device="cpu")

        env = DogfightEnv(cfg.env)
        rewards, wins = [], 0
        for _ in range(20):
            obs, _ = env.reset()
            done = False
            ep_r = 0.0
            while not done:
                norm_obs = stats.normalize(obs).reshape(1, -1)
                action, _ = model.predict(norm_obs, deterministic=True)
                obs, r, term, trunc, info = env.step(action[0])
                ep_r += r
                done = term or trunc
            rewards.append(ep_r)
            if info["my_score"] > info["opp_score"]:
                wins += 1
        mean_reward = float(np.mean(rewards))
        win_rate = wins / 20

        version = pool.add(str(model_path), str(vecnorm_path), mean_reward, win_rate,
                            note="stage-a seed")
        print(f"Havuz tohumlandi: v{version}, mean_reward={mean_reward:.2f}, win_rate={win_rate:.2f}")
        return

    cfg = load_dogfight_config(args.config)
    timesteps = args.timesteps or cfg.train.timesteps
    n_envs = args.n_envs or cfg.train.n_envs
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    venv = make_dogfight_training_vec_env(
        cfg.env, n_envs=n_envs, stage=args.stage, pool_dir=args.pool,
        training=True, norm_reward=True,
    )

    policy_kwargs = build_policy_kwargs(cfg.ppo)
    model = PPO(
        cfg.ppo.policy, venv,
        n_steps=cfg.ppo.n_steps, batch_size=cfg.ppo.batch_size, n_epochs=cfg.ppo.n_epochs,
        gamma=cfg.ppo.gamma, gae_lambda=cfg.ppo.gae_lambda, clip_range=cfg.ppo.clip_range,
        learning_rate=cfg.ppo.learning_rate, ent_coef=cfg.ppo.ent_coef,
        policy_kwargs=policy_kwargs, verbose=1, device="cpu",
        tensorboard_log=str(out / "tb"),
    )

    ckpt_cb = CheckpointCallback(save_freq=max(20_000 // n_envs, 1),
                                  save_path=str(out / "ckpt"), name_prefix="ppo")
    standoff_cb = StandoffCurriculumCallback(
        cfg.env.standoff_weight_start, cfg.env.standoff_weight_end, cfg.env.standoff_ramp_steps
    )
    snapshot_cb = PeriodicSnapshotCallback(out, save_freq=max(args.snapshot_freq // n_envs, 1))
    callbacks = [ckpt_cb, standoff_cb, snapshot_cb]

    if args.stage == "b":
        if not args.pool:
            raise ValueError("--stage b icin --pool gerekli")
        pool = CheckpointPool(args.pool)
        if len(pool) == 0:
            raise ValueError("Havuz bos - once --seed-pool calistirin")
        promo_cb = PromotionCallback(pool, cfg.env, cfg.promotion, out)
        callbacks.append(promo_cb)

    model.learn(total_timesteps=timesteps, callback=callbacks)

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print(f"Egitim tamamlandi: {out}")


if __name__ == "__main__":
    main()

Overwriting /content/repo/src/drone_rl/dogfight/train.py


In [23]:
%%writefile /content/repo/src/drone_rl/dogfight/realtime_dogfight_server.py
import asyncio
import json
import os

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse

from drone_rl.dogfight.config import load_dogfight_config
from drone_rl.dogfight.dogfight_env import DogfightEnv, NormalizerStats
from drone_rl.dogfight.env_factory import load_opponent_controller
from drone_rl.dogfight.checkpoint_pool import CheckpointPool

app = FastAPI()
STATE = {
    "html_path": None, "env": None, "training_controller": None,
    "demo_max_steps": None, "live_snapshot_dir": None, "pool_dir": None,
    "reload_interval_s": 15.0, "_last_training_mtime": None,
    "_last_pool_version": None, "_last_event": None,
}


@app.get("/")
def index():
    return FileResponse(STATE["html_path"])


def _check_and_reload_training():
    model_path = os.path.join(STATE["live_snapshot_dir"], "model.zip")
    vecnorm_path = os.path.join(STATE["live_snapshot_dir"], "vecnormalize.pkl")
    if not (os.path.exists(model_path) and os.path.exists(vecnorm_path)):
        return
    mtime = os.path.getmtime(model_path)
    if STATE["_last_training_mtime"] is not None and mtime <= STATE["_last_training_mtime"]:
        return

    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv
    from stable_baselines3.common.monitor import Monitor

    dummy = DummyVecEnv([lambda: Monitor(DogfightEnv(STATE["env"].cfg))])
    vecnorm = VecNormalize.load(vecnorm_path, dummy)
    stats = NormalizerStats(vecnorm)
    model = PPO.load(model_path, device="cpu")

    STATE["training_controller"] = _TrainingSelfController(model, stats)
    STATE["_last_training_mtime"] = mtime
    STATE["_last_event"] = {"type": "training_updated"}
    print("[reload] TRAINING modeli guncellendi")


def _check_and_reload_best():
    pool = CheckpointPool(STATE["pool_dir"])
    if len(pool) == 0:
        return
    version = pool.entries[-1]["version"]
    if STATE["_last_pool_version"] is not None and version <= STATE["_last_pool_version"]:
        return

    new_controller = load_opponent_controller(*pool.latest())
    STATE["env"].set_opponent_controller(new_controller)
    STATE["_last_pool_version"] = version
    STATE["_last_event"] = {"type": "best_updated", "version": version}
    print(f"[reload] BEST modeli guncellendi -> v{version}")


async def reload_watcher():
    while True:
        await asyncio.sleep(STATE["reload_interval_s"])
        try:
            if STATE["env"] is not None:
                _check_and_reload_training()
                _check_and_reload_best()
        except Exception as e:
            print(f"[reload] hata (atlaniyor): {e}")


@app.on_event("startup")
async def _on_startup():
    asyncio.create_task(reload_watcher())


@app.websocket("/ws")
async def dogfight_loop(websocket: WebSocket):
    await websocket.accept()
    env: DogfightEnv = STATE["env"]
    demo_max_steps = STATE["demo_max_steps"]

    obs, _ = env.reset()
    total_steps = 0
    finished = False
    cumulative_my_score = 0
    cumulative_opp_score = 0
    episode_count = 0
    last_reset_reason = None
    info = None

    try:
        while True:
            if not finished:
                training_action = STATE["training_controller"].compute_action_for_self(env)
                obs, reward, terminated, truncated, info = env.step(training_action)
                total_steps += 1

                if terminated or truncated:
                    cumulative_my_score += info["my_score"]
                    cumulative_opp_score += info["opp_score"]
                    episode_count += 1
                    last_reset_reason = info.get("reset_reason")
                    obs, _ = env.reset()

                if demo_max_steps and total_steps >= demo_max_steps:
                    finished = True

            payload = {
                "self_pos": info["self_pos"], "opp_pos": info["opp_pos"],
                "self_attitude": info["self_attitude"], "opp_attitude": info["opp_attitude"],
                "self_hdot_fps": info["self_hdot_fps"], "opp_hdot_fps": info["opp_hdot_fps"],
                "range_ft": info["range_ft"], "closing_fps": info["closing_fps"],
                "opp_in_my_cone": info["opp_in_my_cone"], "me_in_opp_cone": info["me_in_opp_cone"],
                "align_reward": info["align_reward"], "exposure_penalty": info["exposure_penalty"],
                "standoff_penalty": info["standoff_penalty"], "control_penalty": info["control_penalty"],
                "opp_align_reward": info["opp_align_reward"], "opp_exposure_penalty": info["opp_exposure_penalty"],
                "opp_standoff_penalty": info["opp_standoff_penalty"], "opp_control_penalty": info["opp_control_penalty"],
                "my_score": cumulative_my_score + info["my_score"],
                "opp_score": cumulative_opp_score + info["opp_score"],
                "episode_count": episode_count, "last_reset_reason": last_reset_reason,
                "cone_half_angle_deg": env.cfg.cone_half_angle_deg,
                "cone_range_ft": env.cfg.cone_range_ft,
                "step": total_steps, "finished": finished,
            }
            if STATE["_last_event"] is not None:
                payload["event"] = STATE["_last_event"]
                STATE["_last_event"] = None

            await websocket.send_text(json.dumps(payload))
            await asyncio.sleep(env.control_dt if not finished else 1.0)

    except WebSocketDisconnect:
        pass


class _TrainingSelfController:
    def __init__(self, model, stats):
        self.model = model
        self.stats = stats

    def compute_action_for_self(self, env: DogfightEnv):
        obs = env._get_obs_for(env.fdm_self, env.fdm_opp, env.prev_action_self)
        norm_obs = self.stats.normalize(obs).reshape(1, -1)
        action, _ = self.model.predict(norm_obs, deterministic=True)
        return action[0]


def start_server(live_snapshot_dir: str, pool_dir: str, config_path: str, html_path: str,
                  demo_max_steps: int = None, reload_interval_s: float = 15.0, port: int = 8020):
    import threading
    import uvicorn
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv
    from stable_baselines3.common.monitor import Monitor

    cfg = load_dogfight_config(config_path)

    model_path = os.path.join(live_snapshot_dir, "model.zip")
    vecnorm_path = os.path.join(live_snapshot_dir, "vecnormalize.pkl")
    if not (os.path.exists(model_path) and os.path.exists(vecnorm_path)):
        raise FileNotFoundError(f"Henuz bir egitim anlik goruntusu yok: {live_snapshot_dir}")

    pool = CheckpointPool(pool_dir)
    if len(pool) == 0:
        raise FileNotFoundError(f"Havuz bos: {pool_dir}. Once --seed-pool calistirin.")

    opp_controller = load_opponent_controller(*pool.latest())
    env = DogfightEnv(cfg.env, opponent_controller=opp_controller)

    dummy = DummyVecEnv([lambda: Monitor(DogfightEnv(cfg.env))])
    vecnorm = VecNormalize.load(vecnorm_path, dummy)
    stats = NormalizerStats(vecnorm)
    model = PPO.load(model_path, device="cpu")

    STATE["env"] = env
    STATE["training_controller"] = _TrainingSelfController(model, stats)
    STATE["html_path"] = html_path
    STATE["demo_max_steps"] = demo_max_steps
    STATE["live_snapshot_dir"] = live_snapshot_dir
    STATE["pool_dir"] = pool_dir
    STATE["reload_interval_s"] = reload_interval_s
    STATE["_last_training_mtime"] = os.path.getmtime(model_path)
    STATE["_last_pool_version"] = pool.entries[-1]["version"]
    STATE["_last_event"] = None

    thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
        daemon=True,
    )
    thread.start()
    print(f"Dogfight sunucusu baslatildi (port {port}). Her {reload_interval_s}s kontrol edilecek.")

Overwriting /content/repo/src/drone_rl/dogfight/realtime_dogfight_server.py


In [24]:
%%writefile /content/repo/configs/dogfight_stage_a.yaml
env:
  episode_seconds: 45.0
  physics_hz: 240
  control_hz: 20
  hover_throttle: 0.420
  throttle_range: 0.25
  roll_authority: 0.6
  pitch_authority: 0.6
  yaw_authority: 0.45
  control_surface_tau_s: 0.08
  cone_half_angle_deg: 30.0
  cone_range_ft: 70.0
  standoff_target_ft: 40.0
  standoff_weight_start: 0.02
  standoff_weight_end: 0.06
  standoff_ramp_steps: 200000
  standoff_penalty_cap: 2.0
  reward_align_weight: 0.20
  reward_exposure_weight: 0.15
  reward_cone_hold: 0.08
  reward_tilt_weight: 0.03
  reward_spin_weight: 0.06
  reward_yawrate_weight: 0.04
  reward_jerk_weight: 0.05
  opponent_fault_bonus: 5.0
  crash_penalty: 30.0
  crash_min_alt_ft: 5.0
  crash_max_alt_ft: 250.0
  crash_max_tilt_rad: 0.7
  crash_max_yawrate_rps: 20.0
  max_horizontal_range_ft: 220.0
  min_separation_ft: 10.0
  base_altitude_ft: 150.0
  altitude_jitter_ft: 15.0
  spawn_range_min_ft: 60.0
  spawn_range_max_ft: 150.0
ppo:
  policy: MlpPolicy
  n_steps: 2048
  batch_size: 256
  n_epochs: 10
  gamma: 0.99
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  ent_coef: 0.0
  net_arch_pi: [128, 128]
  net_arch_vf: [128, 128]
  activation_fn: tanh
train:
  timesteps: 400000
  n_envs: 4
promotion:
  eval_freq: 30000
  n_eval_episodes: 30
  win_rate_threshold: 0.55
  mean_reward_improve_pct: 8.0
  consecutive_passes_required: 3

Overwriting /content/repo/configs/dogfight_stage_a.yaml


In [25]:
%%writefile /content/repo/configs/dogfight_stage_b.yaml
env:
  episode_seconds: 45.0
  physics_hz: 240
  control_hz: 20
  hover_throttle: 0.420
  throttle_range: 0.25
  roll_authority: 0.6
  pitch_authority: 0.6
  yaw_authority: 0.45
  control_surface_tau_s: 0.08
  cone_half_angle_deg: 30.0
  cone_range_ft: 70.0
  standoff_target_ft: 40.0
  standoff_weight_start: 0.05
  standoff_weight_end: 0.09
  standoff_ramp_steps: 300000
  standoff_penalty_cap: 2.0
  reward_align_weight: 0.20
  reward_exposure_weight: 0.15
  reward_cone_hold: 0.08
  reward_tilt_weight: 0.03
  reward_spin_weight: 0.06
  reward_yawrate_weight: 0.04
  reward_jerk_weight: 0.05
  opponent_fault_bonus: 5.0
  crash_penalty: 30.0
  crash_min_alt_ft: 5.0
  crash_max_alt_ft: 250.0
  crash_max_tilt_rad: 0.7
  crash_max_yawrate_rps: 20.0
  max_horizontal_range_ft: 220.0
  min_separation_ft: 10.0
  base_altitude_ft: 150.0
  altitude_jitter_ft: 15.0
  spawn_range_min_ft: 60.0
  spawn_range_max_ft: 150.0
  opponent_latest_prob: 0.7
ppo:
  policy: MlpPolicy
  n_steps: 2048
  batch_size: 256
  n_epochs: 10
  gamma: 0.99
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  ent_coef: 0.0
  net_arch_pi: [128, 128]
  net_arch_vf: [128, 128]
  activation_fn: tanh
train:
  timesteps: 1000000
  n_envs: 4
promotion:
  eval_freq: 30000
  n_eval_episodes: 30
  win_rate_threshold: 0.55
  mean_reward_improve_pct: 8.0
  consecutive_passes_required: 3

Overwriting /content/repo/configs/dogfight_stage_b.yaml


In [26]:
%%writefile /content/repo/main.py
#!/usr/bin/env python3
import argparse
import sys
import time
import webbrowser
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent
SRC_DIR = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

CONFIGS_DIR = REPO_ROOT / "configs"
RUNS_DIR = REPO_ROOT / "runs"
HTML_PATH = REPO_ROOT / "dogfightSim_realtime.html"


def cmd_train_a(args):
    from drone_rl.dogfight import train as train_mod
    out = args.out or str(RUNS_DIR / "dogfight_stage_a")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_a.yaml")
    argv = ["train.py", "--stage", "a", "--config", config, "--out", out]
    if args.timesteps:
        argv += ["--timesteps", str(args.timesteps)]
    if args.n_envs:
        argv += ["--n-envs", str(args.n_envs)]
    argv += ["--snapshot-freq", str(args.snapshot_freq)]
    sys.argv = argv
    train_mod.main()


def cmd_seed_pool(args):
    from drone_rl.dogfight import train as train_mod
    from_run = args.from_run or str(RUNS_DIR / "dogfight_stage_a")
    pool = args.pool or str(RUNS_DIR / "dogfight_pool")
    sys.argv = ["train.py", "--seed-pool", "--from", from_run, "--pool", pool]
    train_mod.main()


def cmd_train_b(args):
    from drone_rl.dogfight import train as train_mod
    out = args.out or str(RUNS_DIR / "dogfight_stage_b")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_b.yaml")
    pool = args.pool or str(RUNS_DIR / "dogfight_pool")
    argv = ["train.py", "--stage", "b", "--config", config, "--out", out, "--pool", pool]
    if args.timesteps:
        argv += ["--timesteps", str(args.timesteps)]
    if args.n_envs:
        argv += ["--n-envs", str(args.n_envs)]
    argv += ["--snapshot-freq", str(args.snapshot_freq)]
    sys.argv = argv
    train_mod.main()


def cmd_demo(args):
    from drone_rl.dogfight.realtime_dogfight_server import start_server
    live_snapshot_dir = args.live_snapshot_dir or str(RUNS_DIR / "dogfight_stage_b" / "live_snapshot")
    pool_dir = args.pool or str(RUNS_DIR / "dogfight_pool")
    config = args.config or str(CONFIGS_DIR / "dogfight_stage_b.yaml")
    html_path = args.html or str(HTML_PATH)

    start_server(
        live_snapshot_dir=live_snapshot_dir, pool_dir=pool_dir,
        config_path=config, html_path=html_path,
        demo_max_steps=args.max_steps, reload_interval_s=args.reload_interval,
        port=args.port,
    )

    url = f"http://localhost:{args.port}"
    print(f"\nSunucu hazir: {url}")
    if not args.no_browser:
        try:
            webbrowser.open(url)
        except Exception:
            pass
    print("Durdurmak icin Ctrl+C.\n")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nKapatiliyor.")


def build_parser():
    ap = argparse.ArgumentParser(prog="main.py")
    sub = ap.add_subparsers(dest="command", required=True)

    p_a = sub.add_parser("train-a")
    p_a.add_argument("--config", type=str, default=None)
    p_a.add_argument("--out", type=str, default=None)
    p_a.add_argument("--timesteps", type=int, default=None)
    p_a.add_argument("--n-envs", type=int, default=None)
    p_a.add_argument("--snapshot-freq", type=int, default=10000)
    p_a.set_defaults(func=cmd_train_a)

    p_seed = sub.add_parser("seed-pool")
    p_seed.add_argument("--from", dest="from_run", type=str, default=None)
    p_seed.add_argument("--pool", type=str, default=None)
    p_seed.set_defaults(func=cmd_seed_pool)

    p_b = sub.add_parser("train-b")
    p_b.add_argument("--config", type=str, default=None)
    p_b.add_argument("--out", type=str, default=None)
    p_b.add_argument("--pool", type=str, default=None)
    p_b.add_argument("--timesteps", type=int, default=None)
    p_b.add_argument("--n-envs", type=int, default=None)
    p_b.add_argument("--snapshot-freq", type=int, default=10000)
    p_b.set_defaults(func=cmd_train_b)

    p_demo = sub.add_parser("demo")
    p_demo.add_argument("--live-snapshot-dir", dest="live_snapshot_dir", type=str, default=None)
    p_demo.add_argument("--pool", type=str, default=None)
    p_demo.add_argument("--config", type=str, default=None)
    p_demo.add_argument("--html", type=str, default=None)
    p_demo.add_argument("--port", type=int, default=8020)
    p_demo.add_argument("--max-steps", dest="max_steps", type=int, default=None)
    p_demo.add_argument("--reload-interval", dest="reload_interval", type=float, default=15.0)
    p_demo.add_argument("--no-browser", action="store_true")
    p_demo.set_defaults(func=cmd_demo)

    return ap


def main():
    parser = build_parser()
    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()

Writing /content/repo/main.py


In [27]:
%%writefile /content/repo/requirements.txt
jsbsim
torch
stable-baselines3
gymnasium
fastapi
uvicorn
pyyaml
numpy

Overwriting /content/repo/requirements.txt


In [28]:
%%writefile /content/repo/dogfightSim_realtime.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>F450 Dogfight — Self-Play Simulator</title>
<meta name="viewport" content="width=device-width, initial-scale=1" />
<style>
  :root{
    --bg-0:#0b0f14; --bg-1:#111820; --bg-2:#161f29; --line:#26323e;
    --ink-0:#e8eef3; --ink-1:#9fb0bd; --ink-2:#5f7180;
    --training:#4fd1c5; --best:#e0a05a; --danger:#e0596a;
    --mono: "JetBrains Mono","SF Mono",Consolas,monospace;
    --sans: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
  }
  *{box-sizing:border-box;}
  html,body{ margin:0; height:100%; background:var(--bg-0); color:var(--ink-0); font-family:var(--sans); }
  #app{ display:flex; flex-direction:column; height:100vh; }

  header{
    display:flex; align-items:center; gap:14px; padding:12px 18px;
    background:var(--bg-1); border-bottom:1px solid var(--line); flex-wrap:wrap;
  }
  header h1{ font-size:15px; font-weight:600; margin:0; white-space:nowrap; }
  header h1 span{ color:var(--training); }

  .toggle-group{ display:flex; border:1px solid var(--line); border-radius:6px; overflow:hidden; }
  .toggle-group button{
    background:var(--bg-2); color:var(--ink-1); border:none; padding:7px 14px;
    font-size:12px; font-family:var(--mono); cursor:pointer;
  }
  .toggle-group button.active{ background:var(--training); color:#04211e; font-weight:700; }

  .conn-status{
    margin-left:auto; display:flex; align-items:center; gap:8px;
    font-family:var(--mono); font-size:12px;
  }
  .conn-dot{ width:8px; height:8px; border-radius:50%; background:var(--ink-2); }
  .conn-dot.connected{ background:var(--training); }
  .conn-dot.disconnected{ background:var(--danger); }

  main{ flex:1; display:grid; grid-template-columns: 1fr 300px; min-height:0; }
  #viewport-wrap{ position:relative; }
  #three-canvas{ width:100%; height:100%; display:block; }

  #scoreboard{
    position:absolute; top:14px; left:50%; transform:translateX(-50%);
    display:flex; gap:0; background:rgba(17,24,32,.88); border:1px solid var(--line);
    border-radius:10px; overflow:hidden; font-family:var(--mono); backdrop-filter: blur(4px);
  }
  .score-cell{ padding:10px 22px; text-align:center; min-width:110px; }
  .score-cell.training{ border-right:1px solid var(--line); }
  .score-cell .label{ font-size:10px; color:var(--ink-2); letter-spacing:.5px; }
  .score-cell.training .label{ color:var(--training); }
  .score-cell.best .label{ color:var(--best); }
  .score-cell .value{ font-size:26px; font-weight:700; margin-top:2px; }
  .score-cell.scoring{ animation: flash 0.5s ease-in-out; }
  @keyframes flash{ 0%,100%{background:transparent;} 50%{background:rgba(255,255,255,.15);} }

  #hud{
    position:absolute; bottom:14px; left:14px;
    background:rgba(17,24,32,.85); border:1px solid var(--line);
    border-radius:8px; padding:10px 14px; font-family:var(--mono); font-size:12px;
    line-height:1.7; color:var(--ink-0); backdrop-filter: blur(4px); min-width:190px;
  }
  #hud .row{ display:flex; justify-content:space-between; gap:18px; }
  #hud .row .k{ color:var(--ink-2); }
  #hud .cone-status.active{ color:var(--danger); font-weight:700; }
  #hud .reset-reason{ color:var(--best); }

  #finished-overlay{
    position:absolute; inset:0; display:none; align-items:center; justify-content:center;
    background:rgba(11,15,20,.75); backdrop-filter: blur(3px);
  }
  #finished-overlay.show{ display:flex; }
  #finished-overlay .box{
    background:var(--bg-1); border:1px solid var(--line); border-radius:12px;
    padding:28px 40px; text-align:center;
  }
  #finished-overlay h2{ margin:0 0 8px; color:var(--training); }
  #finished-overlay p{ margin:0; color:var(--ink-1); font-family:var(--mono); font-size:13px; }

  aside#compare{
    background:var(--bg-1); border-left:1px solid var(--line);
    padding:14px; overflow:auto; font-family:var(--mono); font-size:11px;
  }
  aside#compare h3{ margin:0 0 10px; font-size:11px; color:var(--ink-2); font-weight:600; letter-spacing:.5px; }
  .compare-table{ width:100%; border-collapse:collapse; }
  .compare-table th{
    text-align:right; padding:5px 6px; font-size:10px; font-weight:700;
    border-bottom:1px solid var(--line);
  }
  .compare-table th.metric-col{ text-align:left; color:var(--ink-2); font-weight:400; }
  .compare-table th.training-col{ color:var(--training); }
  .compare-table th.best-col{ color:var(--best); }
  .compare-table td{ padding:5px 6px; text-align:right; border-bottom:1px solid rgba(38,50,62,.5); }
  .compare-table td.metric-name{ text-align:left; color:var(--ink-2); }
  .compare-table tr.section-row td{
    padding-top:12px; color:var(--ink-1); font-weight:700; border-bottom:1px solid var(--line);
  }

  /* -- YENI: canli guncelleme bildirimleri (toast) -- */
  #toast-container{
    position:absolute; top:14px; right:14px; display:flex; flex-direction:column;
    gap:8px; align-items:flex-end; pointer-events:none; z-index:20;
  }
  .toast{
    background:rgba(17,24,32,.95); border:1px solid var(--line); border-left:3px solid var(--training);
    border-radius:8px; padding:10px 16px; font-family:var(--mono); font-size:12px;
    color:var(--ink-0); box-shadow:0 4px 14px rgba(0,0,0,.4);
    animation: toast-in 0.3s ease-out, toast-out 0.4s ease-in 4.6s forwards;
    max-width:280px;
  }
  .toast.best{ border-left-color:var(--best); }
  .toast .toast-title{ font-weight:700; margin-bottom:2px; }
  .toast.training .toast-title{ color:var(--training); }
  .toast.best .toast-title{ color:var(--best); }
  @keyframes toast-in{ from{ opacity:0; transform:translateX(20px);} to{ opacity:1; transform:translateX(0);} }
  @keyframes toast-out{ from{ opacity:1; } to{ opacity:0; transform:translateX(20px);} }
</style>
</head>
<body>
<div id="app">
  <header>
    <h1>F450 <span>Dogfight</span> — Self-Play</h1>
    <div class="toggle-group">
      <button id="btn-3d" class="active">3D</button>
      <button id="btn-2d">2D (top-down)</button>
    </div>
    <div class="conn-status">
      <span class="conn-dot" id="conn-dot"></span>
      <span id="conn-text">connecting…</span>
    </div>
  </header>

  <main>
    <div id="viewport-wrap">
      <canvas id="three-canvas"></canvas>

      <div id="toast-container"></div>

      <div id="scoreboard">
        <div class="score-cell training" id="cell-training">
          <div class="label">TRAINING</div>
          <div class="value" id="score-training">0</div>
        </div>
        <div class="score-cell best" id="cell-best">
          <div class="label">BEST</div>
          <div class="value" id="score-best">0</div>
        </div>
      </div>

      <div id="hud">
        <div class="row"><span class="k">range</span><span class="v" id="hud-range">–</span></div>
        <div class="row"><span class="k">step</span><span class="v" id="hud-step">–</span></div>
        <div class="row"><span class="k">episode</span><span class="v" id="hud-episode">–</span></div>
        <div class="row"><span class="k">son reset</span><span class="v reset-reason" id="hud-reset-reason">–</span></div>
        <div class="row"><span class="k">TRAINING → cone</span><span class="v cone-status" id="hud-cone-mine">–</span></div>
        <div class="row"><span class="k">BEST → cone</span><span class="v cone-status" id="hud-cone-opp">–</span></div>
      </div>

      <div id="finished-overlay">
        <div class="box">
          <h2>SIMULATION COMPLETE</h2>
          <p id="finished-summary">–</p>
        </div>
      </div>
    </div>

    <aside id="compare">
      <h3>CANLI KARŞILAŞTIRMA</h3>
      <table class="compare-table">
        <thead>
          <tr>
            <th class="metric-col">metrik</th>
            <th class="training-col">TRAINING</th>
            <th class="best-col">BEST</th>
          </tr>
        </thead>
        <tbody>
          <tr class="section-row"><td colspan="3">Kinematik</td></tr>
          <tr><td class="metric-name">roll (°)</td><td id="cmp-roll-t">–</td><td id="cmp-roll-b">–</td></tr>
          <tr><td class="metric-name">pitch (°)</td><td id="cmp-pitch-t">–</td><td id="cmp-pitch-b">–</td></tr>
          <tr><td class="metric-name">yaw (°)</td><td id="cmp-yaw-t">–</td><td id="cmp-yaw-b">–</td></tr>
          <tr><td class="metric-name">irtifa (ft)</td><td id="cmp-alt-t">–</td><td id="cmp-alt-b">–</td></tr>
          <tr><td class="metric-name">dikey hız (ft/s)</td><td id="cmp-hdot-t">–</td><td id="cmp-hdot-b">–</td></tr>
          <tr class="section-row"><td colspan="3">Ödül bileşenleri (adım başı)</td></tr>
          <tr><td class="metric-name">align</td><td id="cmp-align-t">–</td><td id="cmp-align-b">–</td></tr>
          <tr><td class="metric-name">exposure</td><td id="cmp-exposure-t">–</td><td id="cmp-exposure-b">–</td></tr>
          <tr><td class="metric-name">standoff</td><td id="cmp-standoff-t">–</td><td id="cmp-standoff-b">–</td></tr>
          <tr><td class="metric-name">control</td><td id="cmp-control-t">–</td><td id="cmp-control-b">–</td></tr>
        </tbody>
      </table>
    </aside>
  </main>
</div>

<script type="importmap">
{
  "imports": {
    "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
    "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
  }
}
</script>

<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

const ARENA_HALF_FT = 220;

const canvas = document.getElementById("three-canvas");
const connDot = document.getElementById("conn-dot");
const connText = document.getElementById("conn-text");
const scoreTrainingEl = document.getElementById("score-training");
const scoreBestEl = document.getElementById("score-best");
const cellTraining = document.getElementById("cell-training");
const cellBest = document.getElementById("cell-best");
const toastContainer = document.getElementById("toast-container");
const hud = {
  range: document.getElementById("hud-range"),
  step: document.getElementById("hud-step"),
  episode: document.getElementById("hud-episode"),
  resetReason: document.getElementById("hud-reset-reason"),
  coneMine: document.getElementById("hud-cone-mine"),
  coneOpp: document.getElementById("hud-cone-opp"),
};
const cmp = {
  rollT: document.getElementById("cmp-roll-t"), rollB: document.getElementById("cmp-roll-b"),
  pitchT: document.getElementById("cmp-pitch-t"), pitchB: document.getElementById("cmp-pitch-b"),
  yawT: document.getElementById("cmp-yaw-t"), yawB: document.getElementById("cmp-yaw-b"),
  altT: document.getElementById("cmp-alt-t"), altB: document.getElementById("cmp-alt-b"),
  hdotT: document.getElementById("cmp-hdot-t"), hdotB: document.getElementById("cmp-hdot-b"),
  alignT: document.getElementById("cmp-align-t"), alignB: document.getElementById("cmp-align-b"),
  exposureT: document.getElementById("cmp-exposure-t"), exposureB: document.getElementById("cmp-exposure-b"),
  standoffT: document.getElementById("cmp-standoff-t"), standoffB: document.getElementById("cmp-standoff-b"),
  controlT: document.getElementById("cmp-control-t"), controlB: document.getElementById("cmp-control-b"),
};
const finishedOverlay = document.getElementById("finished-overlay");
const finishedSummary = document.getElementById("finished-summary");

const RESET_REASON_LABELS = {
  collision: "ÇARPIŞMA",
  self_crash: "TRAINING düştü/sınır dışı",
  opponent_crash: "BEST düştü/sınır dışı",
  timeout: "süre doldu",
};

/* -------------------------------------------------------------- */
/* YENI: canli guncelleme bildirimleri (toast)                      */
/* -------------------------------------------------------------- */
function showToast(kind, title, detail) {
  const el = document.createElement("div");
  el.className = `toast ${kind}`;
  el.innerHTML = `<div class="toast-title">${title}</div><div>${detail}</div>`;
  toastContainer.appendChild(el);
  setTimeout(() => el.remove(), 5000);
}

function handleEvent(event) {
  if (!event) return;
  if (event.type === "training_updated") {
    showToast("training", "TRAINING güncellendi", "Eğitim ilerledi, yeni model yüklendi.");
  } else if (event.type === "best_updated") {
    showToast("best", "BEST güncellendi", `Promotion gerçekleşti → v${event.version}`);
  }
}

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(50, 1, 0.1, 3000);
camera.up.set(0, 0, 1);

const renderer = new THREE.WebGLRenderer({ canvas, antialias: true });
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.dampingFactor = 0.08;
controls.target.set(0, 0, 150);
camera.position.set(300, -300, 350);
controls.update();

scene.add(new THREE.AmbientLight(0xffffff, 0.6));
const sun = new THREE.DirectionalLight(0xffffff, 0.9);
sun.position.set(200, -150, 300);
scene.add(sun);

const grid = new THREE.GridHelper(ARENA_HALF_FT * 2 * 1.15, 40, 0x2a3644, 0x1a232c);
grid.rotation.x = Math.PI / 2;
scene.add(grid);

/* -------------------------------------------------------------- */
/* Basit "X" ucak ikonu - burun isaretcisi YOK, koni yonu +X'e      */
/* isaret edecek sekilde duzeltildi.                                 */
/* -------------------------------------------------------------- */
const ARM_ANGLES_DEG = [45, 135, 225, 315];
const ARM_LEN = 2.6, ARM_W = 0.35, HUB_R = 0.5;

function buildAircraftIcon(bodyColorHex) {
  const group = new THREE.Group();

  const mesh3d = new THREE.Group();
  const bodyMat3d = new THREE.MeshStandardMaterial({ color: bodyColorHex, metalness: 0.2, roughness: 0.6 });
  const armGeom3d = new THREE.BoxGeometry(ARM_LEN, ARM_W, ARM_W);
  ARM_ANGLES_DEG.forEach((deg) => {
    const arm = new THREE.Mesh(armGeom3d, bodyMat3d);
    const rad = THREE.MathUtils.degToRad(deg);
    arm.position.set(Math.cos(rad) * (ARM_LEN / 2), Math.sin(rad) * (ARM_LEN / 2), 0);
    arm.rotation.z = rad;
    mesh3d.add(arm);
  });
  mesh3d.add(new THREE.Mesh(new THREE.SphereGeometry(HUB_R, 12, 12), bodyMat3d));
  group.add(mesh3d);

  const mesh2d = new THREE.Group();
  const bodyMat2d = new THREE.MeshBasicMaterial({ color: bodyColorHex, side: THREE.DoubleSide });
  const armGeom2d = new THREE.PlaneGeometry(ARM_LEN, ARM_W);
  ARM_ANGLES_DEG.forEach((deg) => {
    const arm = new THREE.Mesh(armGeom2d, bodyMat2d);
    const rad = THREE.MathUtils.degToRad(deg);
    arm.position.set(Math.cos(rad) * (ARM_LEN / 2), Math.sin(rad) * (ARM_LEN / 2), 0);
    arm.rotation.z = rad;
    mesh2d.add(arm);
  });
  mesh2d.visible = false;
  group.add(mesh2d);

  return { group, mesh3d, mesh2d };
}

function makeCone3d(colorHex, halfAngleDeg, rangeFt) {
  const radius = rangeFt * Math.tan(THREE.MathUtils.degToRad(halfAngleDeg));
  const geom = new THREE.ConeGeometry(radius, rangeFt, 24, 1, true);
  geom.translate(0, -rangeFt / 2, 0);
  geom.rotateZ(Math.PI / 2); // +X (on) yonune isaret eder
  const mat = new THREE.MeshBasicMaterial({
    color: colorHex, transparent: true, opacity: 0.10, side: THREE.DoubleSide, depthWrite: false,
  });
  return new THREE.Mesh(geom, mat);
}

function makeCone2d(colorHex, halfAngleDeg, rangeFt) {
  const halfRad = THREE.MathUtils.degToRad(halfAngleDeg);
  const geom = new THREE.CircleGeometry(rangeFt, 24, -halfRad, halfRad * 2);
  const mat = new THREE.MeshBasicMaterial({
    color: colorHex, transparent: true, opacity: 0.18, side: THREE.DoubleSide, depthWrite: false,
  });
  const mesh = new THREE.Mesh(geom, mat);
  mesh.visible = false;
  return mesh;
}

const trainingIcon = buildAircraftIcon(0x4fd1c5);
scene.add(trainingIcon.group);
const trainingCone3d = makeCone3d(0x4fd1c5, 30, 120);
trainingIcon.mesh3d.add(trainingCone3d);
const trainingCone2d = makeCone2d(0x4fd1c5, 30, 120);
trainingIcon.mesh2d.add(trainingCone2d);

const bestIcon = buildAircraftIcon(0xe0a05a);
scene.add(bestIcon.group);
const bestCone3d = makeCone3d(0xe0a05a, 30, 120);
bestIcon.mesh3d.add(bestCone3d);
const bestCone2d = makeCone2d(0xe0a05a, 30, 120);
bestIcon.mesh2d.add(bestCone2d);

let conesSized = false;
function ensureConeSize(f) {
  if (conesSized) return;
  conesSized = true;
  [trainingCone3d, bestCone3d].forEach((c) => {
    c.geometry.dispose();
    const radius = f.cone_range_ft * Math.tan(THREE.MathUtils.degToRad(f.cone_half_angle_deg));
    c.geometry = new THREE.ConeGeometry(radius, f.cone_range_ft, 24, 1, true);
    c.geometry.translate(0, -f.cone_range_ft / 2, 0);
    c.geometry.rotateZ(Math.PI / 2);
  });
  [trainingCone2d, bestCone2d].forEach((c) => {
    c.geometry.dispose();
    const halfRad = THREE.MathUtils.degToRad(f.cone_half_angle_deg);
    c.geometry = new THREE.CircleGeometry(f.cone_range_ft, 24, -halfRad, halfRad * 2);
  });
}

const trailTraining = [];
const trailBest = [];
const TRAIL_LEN = 200;
let trailLineTraining = new THREE.Line(new THREE.BufferGeometry(),
  new THREE.LineBasicMaterial({ color: 0x4fd1c5, transparent: true, opacity: 0.5 }));
let trailLineBest = new THREE.Line(new THREE.BufferGeometry(),
  new THREE.LineBasicMaterial({ color: 0xe0a05a, transparent: true, opacity: 0.5 }));
scene.add(trailLineTraining);
scene.add(trailLineBest);

function resizeRenderer() {
  const w = canvas.clientWidth, h = canvas.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w / Math.max(h, 1);
  camera.updateProjectionMatrix();
}
window.addEventListener("resize", resizeRenderer);

let is2D = false;
document.getElementById("btn-3d").addEventListener("click", () => setMode(false));
document.getElementById("btn-2d").addEventListener("click", () => setMode(true));

function setMode(twoD) {
  is2D = twoD;
  document.getElementById("btn-3d").classList.toggle("active", !twoD);
  document.getElementById("btn-2d").classList.toggle("active", twoD);

  trainingIcon.mesh3d.visible = !twoD;
  trainingIcon.mesh2d.visible = twoD;
  bestIcon.mesh3d.visible = !twoD;
  bestIcon.mesh2d.visible = twoD;

  if (twoD) {
    controls.minPolarAngle = 0;
    controls.maxPolarAngle = 0.001;
    camera.position.set(controls.target.x, controls.target.y, controls.target.z + 500);
  } else {
    controls.minPolarAngle = 0;
    controls.maxPolarAngle = Math.PI;
  }
  controls.update();
}

let latestFrame = null;
let lastMyScore = 0, lastOppScore = 0;

function connect() {
  const wsProtocol = window.location.protocol === "https:" ? "wss:" : "ws:";
  const ws = new WebSocket(`${wsProtocol}//${window.location.host}/ws`);

  ws.onopen = () => { connDot.className = "conn-dot connected"; connText.textContent = "connected"; };
  ws.onclose = () => {
    connDot.className = "conn-dot disconnected"; connText.textContent = "disconnected — retrying…";
    setTimeout(connect, 1500);
  };
  ws.onerror = () => ws.close();
  ws.onmessage = (msg) => {
    const f = JSON.parse(msg.data);
    if (f.event) handleEvent(f.event);
    latestFrame = f;
  };
}
connect();

function fmt(v, digits = 2) { return (typeof v === "number" ? v.toFixed(digits) : "–"); }

function renderFrame(f) {
  ensureConeSize(f);

  const [sn, se, salt] = f.self_pos;
  const [on, oe, oalt] = f.opp_pos;
  const [sroll, spitch, syaw] = f.self_attitude;
  const [oroll, opitch, oyaw] = f.opp_attitude;

  trainingIcon.group.position.set(sn, se, salt);
  trainingIcon.mesh3d.rotation.set(sroll, spitch, syaw, "XYZ");
  trainingIcon.mesh2d.rotation.set(0, 0, syaw);

  bestIcon.group.position.set(on, oe, oalt);
  bestIcon.mesh3d.rotation.set(oroll, opitch, oyaw, "XYZ");
  bestIcon.mesh2d.rotation.set(0, 0, oyaw);

  trailTraining.push(new THREE.Vector3(sn, se, salt));
  if (trailTraining.length > TRAIL_LEN) trailTraining.shift();
  trailLineTraining.geometry.dispose();
  trailLineTraining.geometry = new THREE.BufferGeometry().setFromPoints(trailTraining);

  trailBest.push(new THREE.Vector3(on, oe, oalt));
  if (trailBest.length > TRAIL_LEN) trailBest.shift();
  trailLineBest.geometry.dispose();
  trailLineBest.geometry = new THREE.BufferGeometry().setFromPoints(trailBest);

  hud.range.textContent = `${f.range_ft.toFixed(1)} ft`;
  hud.step.textContent = f.step;
  hud.episode.textContent = f.episode_count;
  hud.resetReason.textContent = f.last_reset_reason ? (RESET_REASON_LABELS[f.last_reset_reason] || f.last_reset_reason) : "—";
  hud.coneMine.textContent = f.opp_in_my_cone ? "IN CONE ✓" : "—";
  hud.coneMine.classList.toggle("active", f.opp_in_my_cone);
  hud.coneOpp.textContent = f.me_in_opp_cone ? "IN CONE ✓" : "—";
  hud.coneOpp.classList.toggle("active", f.me_in_opp_cone);

  cmp.rollT.textContent = fmt(sroll * 180 / Math.PI, 1);
  cmp.rollB.textContent = fmt(oroll * 180 / Math.PI, 1);
  cmp.pitchT.textContent = fmt(spitch * 180 / Math.PI, 1);
  cmp.pitchB.textContent = fmt(opitch * 180 / Math.PI, 1);
  cmp.yawT.textContent = fmt(((syaw * 180 / Math.PI) + 360) % 360, 1);
  cmp.yawB.textContent = fmt(((oyaw * 180 / Math.PI) + 360) % 360, 1);
  cmp.altT.textContent = fmt(salt, 1);
  cmp.altB.textContent = fmt(oalt, 1);
  cmp.hdotT.textContent = fmt(f.self_hdot_fps, 2);
  cmp.hdotB.textContent = fmt(f.opp_hdot_fps, 2);
  cmp.alignT.textContent = fmt(f.align_reward, 3);
  cmp.alignB.textContent = fmt(f.opp_align_reward, 3);
  cmp.exposureT.textContent = fmt(-f.exposure_penalty, 3);
  cmp.exposureB.textContent = fmt(-f.opp_exposure_penalty, 3);
  cmp.standoffT.textContent = fmt(-f.standoff_penalty, 3);
  cmp.standoffB.textContent = fmt(-f.opp_standoff_penalty, 3);
  cmp.controlT.textContent = fmt(-f.control_penalty, 3);
  cmp.controlB.textContent = fmt(-f.opp_control_penalty, 3);

  scoreTrainingEl.textContent = f.my_score;
  scoreBestEl.textContent = f.opp_score;
  if (f.my_score > lastMyScore) { cellTraining.classList.add("scoring"); setTimeout(() => cellTraining.classList.remove("scoring"), 500); }
  if (f.opp_score > lastOppScore) { cellBest.classList.add("scoring"); setTimeout(() => cellBest.classList.remove("scoring"), 500); }
  lastMyScore = f.my_score; lastOppScore = f.opp_score;

  if (f.finished) {
    finishedOverlay.classList.add("show");
    finishedSummary.textContent = `TRAINING ${f.my_score} — ${f.opp_score} BEST  (${f.episode_count} episode, ${f.step} adım)`;
  }

  if (!is2D) {
    const midX = (sn + on) / 2, midY = (se + oe) / 2, midZ = (salt + oalt) / 2;
    controls.target.lerp(new THREE.Vector3(midX, midY, midZ), 0.03);
  }
}

function animate() {
  requestAnimationFrame(animate);
  if (latestFrame) renderFrame(latestFrame);
  controls.update();
  resizeRenderer();
  renderer.render(scene, camera);
}
requestAnimationFrame(animate);
</script>
</body>
</html>

Overwriting /content/repo/dogfightSim_realtime.html


In [ ]:
# 1- Stage A - scripted rakibe karşı temel taktik
!cd /content/repo/src && python -m drone_rl.dogfight.train --stage a \
  --config ../configs/dogfight_stage_a.yaml --out /content/repo/runs/dogfight_stage_a

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [ ]:
# 2- Havuzu tohumla
!cd /content/repo/src && python -m drone_rl.dogfight.train --seed-pool \
  --from /content/repo/runs/dogfight_stage_a --pool /content/repo/runs/dogfight_pool

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [ ]:

# 3-  Stage B - self-play + promotion
!cd /content/repo/src && python -m drone_rl.dogfight.train --stage b \
  --config ../configs/dogfight_stage_b.yaml --out /content/repo/runs/dogfight_stage_b \
  --pool /content/repo/runs/dogfight_pool


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [29]:
import subprocess
train_a_process = subprocess.Popen(
    ["python", "main.py", "train-a"],
    cwd="/content/repo",
    stdout=open("/content/repo/train_a_log.txt", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Stage A arka planda başladı, PID={train_a_process.pid}")

Stage A arka planda başladı, PID=12435


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [37]:
print("Süreç durumu:", "ÇALIŞIYOR" if train_a_process.poll() is None else f"BİTTİ/ÇÖKTÜ (kod={train_a_process.poll()})")
print("-" * 60)
!tail -40 /content/repo/train_a_log.txt

Süreç durumu: BİTTİ/ÇÖKTÜ (kod=0)
------------------------------------------------------------
|    fps                  | 384         |
|    iterations           | 48          |
|    time_elapsed         | 1022        |
|    total_timesteps      | 393216      |
| train/                  |             |
|    approx_kl            | 0.009749956 |
|    clip_fraction        | 0.106       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.53       |
|    explained_variance   | 0.842       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0166      |
|    n_updates            | 470         |
|    policy_gradient_loss | -0.0115     |
|    std                  | 0.75        |
|    value_loss           | 0.115       |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 134         |
|    ep_rew_mean          | -39.2       |
| time/                

In [38]:
!cd /content/repo && python main.py seed-pool

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [59]:
train_b_process = subprocess.Popen(
    ["python", "main.py", "train-b"],
    cwd="/content/repo",
    stdout=open("/content/repo/train_b_log.txt", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Stage B arka planda başladı, PID={train_b_process.pid}")

Stage B arka planda başladı, PID=19564


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [68]:
print("Süreç durumu:", "ÇALIŞIYOR" if train_b_process.poll() is None else f"BİTTİ/ÇÖKTÜ (kod={train_b_process.poll()})")
print("-" * 60)
!tail -40 /content/repo/train_b_log.txt

Süreç durumu: ÇALIŞIYOR
------------------------------------------------------------
| time/                   |             |
|    fps                  | 313         |
|    iterations           | 9           |
|    time_elapsed         | 234         |
|    total_timesteps      | 73728       |
| train/                  |             |
|    approx_kl            | 0.008745024 |
|    clip_fraction        | 0.09        |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.49       |
|    explained_variance   | 0.638       |
|    learning_rate        | 0.0003      |
|    loss                 | 0.0505      |
|    n_updates            | 80          |
|    policy_gradient_loss | -0.00884    |
|    std                  | 0.952       |
|    value_loss           | 0.14        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 123         |
|    ep_rew_mean          | -46.3

In [71]:
!ls -la /content/repo/runs/dogfight_stage_b/live_snapshot/ 2>&1

total 496
drwxr-xr-x 2 root root   4096 Sep 10 19:51 .
drwxr-xr-x 5 root root   4096 Sep 10 19:50 ..
-rw-r--r-- 1 root root 494192 Sep 10 20:02 model.zip
-rw-r--r-- 1 root root   2093 Sep 10 20:02 vecnormalize.pkl


In [86]:
import sys
sys.path.insert(0, "/content/repo/src")
from drone_rl.dogfight.realtime_dogfight_server import start_server

start_server(
    live_snapshot_dir="/content/repo/runs/dogfight_stage_b/live_snapshot",
    pool_dir="/content/repo/runs/dogfight_pool",
    config_path="/content/repo/configs/dogfight_stage_b.yaml",
    html_path="/content/repo/dogfightSim_realtime.html",
    reload_interval_s=15.0,
    port=8030,
)

Dogfight sunucusu baslatildi (port 8030). Her 15.0s kontrol edilecek.


In [90]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8030)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [56]:
train_b_process.terminate()

In [89]:
!curl -s -o /dev/null -w "HTTP kodu: %{http_code}\n" http://localhost:8030/

HTTP kodu: 200
